### 1. Initialization and read


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length, date_sub, lead


In [0]:
df = spark.table("catalogue_project1.bronze.crm_sales_details")

### 2. Transformations
2.1 Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

2.2 Make sure dates are Date_type

In [0]:
df = (
    df
    .withColumn(
        "sls_order_dt",
        F.when(
            (col("sls_order_dt") == 0) | (length(col("sls_order_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_order_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_ship_dt",
        F.when(
            (col("sls_ship_dt") == 0) | (length(col("sls_ship_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_due_dt",
        F.when(
            (col("sls_due_dt") == 0) | (length(col("sls_due_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_due_dt").cast("string"), "yyyyMMdd"))
    )
)

2.3 Apply business rule: \
sales = quantity + price

In [0]:
df = df.withColumn("sls_price",
    F.when(
        (col("sls_price") <= 0) | (col("sls_price").isNull()),
        F.when(
            col("sls_quantity") != 0,
            col("sls_sales") / col("sls_quantity")
        ).otherwise(None)
    ).otherwise(col("sls_price"))
)

df = df.withColumn("sls_sales",
    F.when(
        (col("sls_sales") <= 0) | (col("sls_sales").isNull()),
        col("sls_quantity") * col("sls_price")
    ).otherwise(col("sls_sales"))        
)

2.4 Rename columns

In [0]:
RENAME_CONFIG = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in RENAME_CONFIG.items():
    df = df.withColumnRenamed(old_name, new_name)

* Intermediate step: check the changes applied correctly.

In [0]:
df.limit(10).display()

### 3. Write into the Silver delta table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("catalogue_project1.silver.crm_sales_details")

* Check that everything went well

In [0]:
%sql
SELECT * FROM catalogue_project1.silver.crm_sales_details LIMIT 10